In [ ]:
# Plant Disease Classification Model - Epic 3

This notebook builds and trains a deep learning model using transfer learning with MobileNetV2 to classify plant diseases.

## Overview
- **Objective**: Train a CNN model to detect and classify plant diseases from images
- **Approach**: Transfer learning using pre-trained MobileNetV2 from ImageNet
- **Dataset**: Plant Diseases Dataset (Augmented)
- **Framework**: TensorFlow/Keras

## Key Activities
1. Import dependencies and configure environment
2. Load and prepare images with data augmentation
3. Build transfer learning model architecture
4. Compile and configure training callbacks
5. Train the model
6. Evaluate and visualize results

In [ ]:
## 1. Setup & Configuration

Import all required libraries and define constants.

In [ ]:
import os
import pathlib
import warnings
from typing import Tuple

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")


In [ ]:
# Configuration constants
IMG_SIZE: Tuple[int, int] = (224, 224)
BATCH_SIZE: int = 32
EPOCHS: int = 20
VALIDATION_SPLIT: float = 0.2
RANDOM_SEED: int = 42

# Set random seeds for reproducibility
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"Image Size: {IMG_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")


## 2. Dataset Path Configuration

Configure paths and validate dataset availability.

In [ ]:
# Define dataset paths relative to the notebook location
base_dir = pathlib.Path("../archive/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)")
train_dir = base_dir / "train"
valid_dir = base_dir / "valid"

# Validate paths exist
for path_name, path_obj in [("Train", train_dir), ("Validation", valid_dir)]:
    if not path_obj.exists():
        raise FileNotFoundError(f"{path_name} directory not found at {path_obj}")
    else:
        num_classes = len(list(path_obj.glob("*")))
        num_images = sum(1 for _ in path_obj.glob("*/*"))
        print(f"✓ {path_name}: {path_obj} ({num_classes} classes, ~{num_images} images)")


## 3. Data Loading and Augmentation

Create data augmentation pipeline and load training/validation data generators.

In [ ]:
# Create data augmentation pipeline for training data
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Validation data: only normalize, no augmentation
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

print("✓ Data augmentation pipelines created")


In [ ]:
# Load training data
train_generator = train_datagen.flow_from_directory(
    str(train_dir),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=RANDOM_SEED
)

# Load validation data
val_generator = val_datagen.flow_from_directory(
    str(valid_dir),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=RANDOM_SEED
)

num_classes = train_generator.num_classes
class_names = list(train_generator.class_indices.keys())

print(f"✓ Training samples: {train_generator.samples}")
print(f"✓ Validation samples: {val_generator.samples}")
print(f"✓ Number of classes: {num_classes}")
print(f"✓ Classes: {', '.join(class_names[:5])}...")  # Show first 5


## 4. Model Architecture

Build transfer learning model using MobileNetV2 as the base.

**Architecture:**
- Pre-trained MobileNetV2 (ImageNet weights)
- Global Average Pooling layer
- Dropout (0.3) for regularization
- Dense output layer with softmax activation

In [ ]:
# Load pre-trained MobileNetV2 model (without top classification layer)
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(*IMG_SIZE, 3)
)

# Freeze base model weights (transfer learning)
base_model.trainable = False

# Build complete model
model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),  # Reduces spatial dimensions
    layers.Dropout(0.3),              # Regularization to prevent overfitting
    layers.Dense(256, activation='relu'),  # Additional hidden layer
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')  # Output layer
])

print(f"✓ Model created with {num_classes} output classes")
print(f"✓ Total parameters: {model.count_params():,}")


In [ ]:
model.summary()


## 5. Model Compilation

Configure optimizer, loss function, and metrics.

In [ ]:
# Compile the model with optimizer, loss, and metrics
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✓ Model compiled successfully")


## 6. Training Configuration

Setup callbacks for better training control:
- **ModelCheckpoint**: Save best model based on validation accuracy
- **ReduceLROnPlateau**: Reduce learning rate when validation loss plateaus
- **EarlyStopping**: Stop training if validation loss doesn't improve

In [ ]:
# Define training callbacks
callbacks = [
    # Save model checkpoint with best validation accuracy
    keras.callbacks.ModelCheckpoint(
        filepath="best_model.keras",
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    
    # Reduce learning rate on validation loss plateau
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    
    # Early stopping if validation loss stops improving
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    
    # TensorBoard logging (optional)
    keras.callbacks.TensorBoard(
        log_dir='./logs',
        histogram_freq=0,
        write_graph=False
    )
]

print("✓ Callbacks configured successfully!")


## 7. Model Training

Train the model with augmented data and validation monitoring.

In [ ]:
# Train the model
print(f"Training model for {EPOCHS} epochs...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("✓ Training completed!")


## 8. Model Evaluation

Save the trained model and evaluate performance on validation data.

In [ ]:
# Save the final model
model.save("plant_disease_model.h5")
print("✓ Model saved as 'plant_disease_model.h5'")

# Evaluate on validation set
print("\nEvaluating model on validation set...")
val_loss, val_accuracy = model.evaluate(val_generator, verbose=1)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")


In [ ]:
# Get predictions on validation set for detailed metrics
val_generator.reset()  # Reset to start from beginning
predictions = model.predict(val_generator, verbose=0)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = val_generator.classes

# Calculate detailed metrics
print("Classification Report:")
print(classification_report(
    true_classes, 
    predicted_classes,
    target_names=class_names
))

# Overall metrics
overall_accuracy = accuracy_score(true_classes, predicted_classes)
print(f"\nOverall Validation Accuracy: {overall_accuracy:.4f}")


## 9. Results Visualization

Visualize training history and model performance metrics.

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], marker='o', label='Training Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], marker='s', label='Validation Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].set_title('Model Accuracy Over Epochs', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], marker='o', label='Training Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], marker='s', label='Validation Loss', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Loss', fontsize=11)
axes[1].set_title('Model Loss Over Epochs', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Plot confusion matrix (sample if dataset is large)
cm = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, 
    annot=False,  # Too many classes for annotations
    cmap='Blues',
    cbar_kws={'label': 'Count'},
    xticklabels=False,
    yticklabels=False
)
plt.xlabel('Predicted Class', fontsize=12)
plt.ylabel('True Class', fontsize=12)
plt.title('Confusion Matrix - Validation Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Confusion Matrix shape: {cm.shape}")


## Summary

Model training and evaluation completed. Key metrics:
- **Training Accuracy**: {:.2%}
- **Validation Accuracy**: {:.2%}
- **Number of Classes**: {}
- **Total Trainable Parameters**: {:,}

The model uses transfer learning with a pre-trained MobileNetV2 backbone, achieving good performance with data augmentation and early stopping to prevent overfitting.